# Stored execution evidence — Dogs vs. Cats Redux

This public evidence copy preserves every textual output cell supplied in `jiaozi.zip`.
The original notebook SHA-256 is `033ceed2f5944eb461e125f26d3f4646c748798f8939a7c838b21f75a2815b91`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Jiaozi Integration Update on Colab

This notebook deletes any existing `/content/Jiaozi` checkout, clones the latest `main` branch, installs dependencies, and runs the integrated Jiaozi pipeline end to end on Colab.

The active flow is:

1. Module 1 parses the natural-language task with an LLM.
2. Module 2 analyzes the HuggingFace dataset.
3. Module 3 retrieves ranked CV model candidates.
4. Module 4 generates runnable training code.
5. The generated project runs the built-in smoke checks.


## Before running

1. In Colab, open **Secrets** and add `OPENAI_API_KEY`.
2. Add `KAGGLE_API_TOKEN` in Colab Secrets and accept the Plant Pathology competition rules on Kaggle before downloading.
3. Select a GPU runtime for real training.
4. Run the cells in order.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

def normalize_repo_url(value: str) -> str:
    value = (value or '').strip()
    markdown_match = re.fullmatch(r'\[(.*?)\]\((https?://[^)]+)\)', value)
    if markdown_match:
        return markdown_match.group(2)
    plain_url_match = re.search(r'https?://\S+', value)
    if plain_url_match:
        return plain_url_match.group(0)
    return value

REPO_URL = normalize_repo_url(os.getenv('JIAOZI_REPO_URL', 'https://github.com/Isso-W/Jiaozi.git'))
REPO_REF = os.getenv('JIAOZI_REPO_REF', 'main')
REPO_DIR = Path('/content/Jiaozi')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

print("REPO_URL =", REPO_URL)
print("REPO_REF =", REPO_REF)

clone_cmd = [
    'git', 'clone',
    '--depth', '1',
    '--branch', REPO_REF,
    REPO_URL,
    str(REPO_DIR),
]

completed = subprocess.run(
    clone_cmd,
    capture_output=True,
    text=True,
)

print("=== git stdout ===")
print(completed.stdout)
print("=== git stderr ===")
print(completed.stderr)

completed.check_returncode()

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("\n=== requirements.txt ===")
print(Path("requirements.txt").read_text()[:4000])

print("\n=== installing requirements ===")
pip_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
    capture_output=True,
    text=True,
)

print(pip_result.stdout)
print(pip_result.stderr)
print("pip return code =", pip_result.returncode)

if pip_result.returncode != 0:
    raise RuntimeError("pip install failed. Check the pip output above.")

print('Jiaozi repository:', REPO_DIR)
print('Jiaozi repo URL:', REPO_URL)
print('Jiaozi ref:', REPO_REF)

pipeline_candidates = list(REPO_DIR.rglob("pipeline.py"))
print("pipeline.py candidates:")
for p in pipeline_candidates:
    print(" -", p)


REPO_URL = https://github.com/muzhi-hac/Jiaozi-rag-wang.git
REPO_REF = rag_wang
=== git stdout ===

=== git stderr ===
Cloning into '/content/Jiaozi'...


=== requirements.txt ===
# Runtime dependencies for the Jiaozi CV Auto-DL pipeline.
# Install with the same Python interpreter used to run tests, for example:
#   /usr/bin/python3 -m pip install -r requirements.txt

chromadb
datasets
networkx
numpy
openai>=2.0.0
pandas
pillow
scikit-learn
sentence-transformers
torch
torchvision
transformers

# Test runner for the module4_agent pytest suite.
pytest


=== installing requirements ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 130.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━

In [2]:
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_secret(name: str, required: bool = False) -> str:
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if not value and required:
        value = getpass(f'{name} (hidden input): ').strip()
    return value.strip()


openai_api_key = read_secret('OPENAI_API_KEY', required=True)
if not openai_api_key:
    raise RuntimeError('OPENAI_API_KEY is required for this notebook.')

os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['JIAOZI_LLM_PROVIDER'] = 'openai'
os.environ['M4_LLM_PROVIDER'] = 'openai'
os.environ.setdefault('M1_OPENAI_MODEL', 'gpt-5.5')
os.environ.setdefault('M4_OPENAI_MODEL', 'gpt-5.5')

kaggle_token = read_secret('KAGGLE_API_TOKEN', required=False)
if kaggle_token:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_path = kaggle_dir / 'access_token'
    token_path.write_text(kaggle_token, encoding='utf-8')
    token_path.chmod(0o600)
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token')
else:
    print('No KAGGLE_API_TOKEN provided. This branch does not need Kaggle for the default run.')

del openai_api_key
del kaggle_token


Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token


## Download the Plant Pathology data

These cells configure the Kaggle CLI, download `plant-pathology-2020-fgvc7`, and extract the competition files under `/content/data`. The pipeline cell later converts the one-hot `train.csv` into a single `label` column that Jiaozi's local CSV loader can consume.


In [3]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

def run_pip(args):
    print("Running:", " ".join([sys.executable, "-m", "pip"] + args))
    result = subprocess.run(
        [sys.executable, "-m", "pip"] + args,
        capture_output=True,
        text=True,
    )
    print("=== pip stdout ===")
    print(result.stdout)
    print("=== pip stderr ===")
    print(result.stderr)
    print("return code =", result.returncode)
    return result

# First upgrade the basic installation tools
run_pip(["install", "--upgrade", "pip", "setuptools", "wheel"])

# Install the Kaggle CLI again, but do not use the -q option.
kaggle_install = run_pip(["install", "--upgrade", "kaggle"])

if kaggle_install.returncode != 0:
    raise RuntimeError("Kaggle CLI install failed. Check pip output above.")

# Read Kaggle access token from Colab Secrets
kaggle_token = userdata.get("KAGGLE_API_TOKEN")
if not kaggle_token:
    raise RuntimeError("Please add KAGGLE_API_TOKEN in Colab Secrets.")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

access_token_path = kaggle_dir / "access_token"
access_token_path.write_text(kaggle_token.strip(), encoding="utf-8")
access_token_path.chmod(0o600)

os.environ["KAGGLE_API_TOKEN"] = kaggle_token.strip()

print("Kaggle access token configured.")

# Check if Kaggle is available
subprocess.run(["kaggle", "--version"], check=False)


Running: /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
=== pip stdout ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.6 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2

=== pip stderr ===
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.

return code = 0
Running: /usr/bin/python3 -m pip install --upgrade kaggle
=== pip stdout

CompletedProcess(args=['kaggle', '--version'], returncode=0)

In [4]:
KAGGLE_COMPETITION = "dogs-vs-cats-redux-kernels-edition"
!kaggle competitions download -c {KAGGLE_COMPETITION} -p /content/data


100% 814M/814M [00:07<00:00, 121MB/s]



In [5]:
from pathlib import Path
import zipfile

DATA_ROOT = Path("/content/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

zip_files = sorted(DATA_ROOT.glob(f"{KAGGLE_COMPETITION}*.zip")) or sorted(DATA_ROOT.glob("*.zip"))
print("Zip files:", zip_files)

if not zip_files:
    raise FileNotFoundError("No .zip file found in /content/data")

zip_path = zip_files[0]
KAGGLE_DATA_DIR = DATA_ROOT / zip_path.stem
KAGGLE_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting:", zip_path)
print("To:", KAGGLE_DATA_DIR)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(KAGGLE_DATA_DIR)

# Some Kaggle competitions contain nested archives. Extract them in place so image
# discovery below can find the real files regardless of packaging.
for nested_zip in sorted(KAGGLE_DATA_DIR.rglob("*.zip")):
    nested_target = nested_zip.with_suffix("")
    nested_target.mkdir(parents=True, exist_ok=True)
    print("Extracting nested:", nested_zip, "->", nested_target)
    with zipfile.ZipFile(nested_zip, "r") as z:
        z.extractall(nested_target)

print("Done.")
print("KAGGLE_DATA_DIR =", KAGGLE_DATA_DIR)
print("Top-level files:")
for p in sorted(KAGGLE_DATA_DIR.iterdir()):
    print(" -", p.name)


Zip files: [PosixPath('/content/data/dogs-vs-cats-redux-kernels-edition.zip')]
Extracting: /content/data/dogs-vs-cats-redux-kernels-edition.zip
To: /content/data/dogs-vs-cats-redux-kernels-edition
Extracting nested: /content/data/dogs-vs-cats-redux-kernels-edition/test.zip -> /content/data/dogs-vs-cats-redux-kernels-edition/test
Extracting nested: /content/data/dogs-vs-cats-redux-kernels-edition/train.zip -> /content/data/dogs-vs-cats-redux-kernels-edition/train
Done.
KAGGLE_DATA_DIR = /content/data/dogs-vs-cats-redux-kernels-edition
Top-level files:
 - sample_submission.csv
 - test
 - test.zip
 - train
 - train.zip


## Mount Google Drive for caches and checkpoints

HuggingFace caches and training checkpoints can live on Drive so a recycled Colab runtime does not force every artifact to be downloaded or trained from scratch.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Everything HuggingFace caches (datasets + hub) lives here, on Drive.
HF_CACHE_DIR = '/content/drive/MyDrive/Jiaozi/hf_cache'
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = os.path.join(HF_CACHE_DIR, 'datasets')

print('HF_HOME =', os.environ['HF_HOME'])
print('First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).')
print('Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.')


Mounted at /content/drive
HF_HOME = /content/drive/MyDrive/Jiaozi/hf_cache
First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).
Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.


## Run the integrated pipeline

This cell runs the integrated Jiaozi flow against the local Kaggle files:

1. Convert Plant Pathology's one-hot labels into a local CSV with `image_id,label`.
2. Run Module 1 from the natural-language `QUERY`.
3. Run Module 2 on sampled real images from the Kaggle folder.
4. Run Module 3 retrieval and optional recommender re-ranking.
5. Run Module 4 code generation with local CSV training metadata.

Set `REAL_TRAINING = True` to generate real-training code and train it in the next cell. With `REAL_TRAINING = False`, Module 4 keeps the generated project in smoke-test mode.


In [7]:
# [replace Cell 11] Dogs vs Cats Redux Kaggle data → Jiaozi Module 1-4

import json
import os
import shutil
import sys
from pathlib import Path

import pandas as pd

REPO_DIR = Path("/content/Jiaozi")
PIPELINE_PATH = REPO_DIR / "pipeline.py"

if not PIPELINE_PATH.exists():
    raise FileNotFoundError(
        f"No pipeline.py found at {PIPELINE_PATH}. Please rerun the git clone cell first."
    )

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Current working directory:", Path.cwd())
print("Using pipeline:", PIPELINE_PATH)

# -----------------------------
# 1. Dogs vs Cats task prompt
# -----------------------------
QUERY = (
    "Binary image classification for cats and dogs. "
    "Images are natural photos of cats and dogs from the Kaggle Dogs vs Cats Redux competition. "
    "The target label is binary: 0 for cat and 1 for dog. "
    "The primary metric is binary log loss, and the model should output calibrated probabilities for the dog class. "
    "Also report accuracy and ROC-AUC if available. "
    "Dataset properties: natural animal images, varied backgrounds, poses, lighting, and image sizes. "
    "Resource constraint: single Colab T4 GPU. "
    "Model constraint: use only publicly accessible pretrained checkpoints that do not require gated access. "
    "The agent should autonomously select the model, preprocessing, augmentation, loss, optimizer, and validation strategy. "
    "The best checkpoint should be selected by validation log loss or validation ROC-AUC."
)

REAL_TRAINING = True
EPOCHS = None
OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")

# -----------------------------
# 2. Locate Kaggle Dogs vs Cats data
# -----------------------------
DATASET_DIR = Path(KAGGLE_DATA_DIR)

print("DATASET_DIR:", DATASET_DIR)
print("Top-level files:")
for p in sorted(DATASET_DIR.iterdir()):
    print(" -", p.name)

# Cell 7 usually extracts nested train.zip into a folder named "train"
candidate_train_dirs = [
    DATASET_DIR / "train",
    DATASET_DIR / "train" / "train",
]

train_dir = None
for d in candidate_train_dirs:
    if d.exists() and list(d.glob("*.jpg")):
        train_dir = d
        break

if train_dir is None:
    matches = []
    for d in DATASET_DIR.rglob("*"):
        if d.is_dir():
            jpgs = list(d.glob("*.jpg"))
            if jpgs and any(p.name.startswith("cat.") or p.name.startswith("dog.") for p in jpgs[:50]):
                matches.append(d)
    if matches:
        train_dir = matches[0]

if train_dir is None:
    raise FileNotFoundError(
        f"Could not find Dogs vs Cats train images under {DATASET_DIR}. "
        "Expected files like cat.0.jpg and dog.0.jpg."
    )

print("Train image dir:", train_dir)
print("Sample train files:", [p.name for p in sorted(train_dir.glob('*.jpg'))[:10]])

# -----------------------------
# 3. Build Jiaozi-friendly train CSV
#    cat.*.jpg -> label 0
#    dog.*.jpg -> label 1
# -----------------------------
rows = []

for p in sorted(train_dir.glob("*.jpg")):
    if p.name.startswith("cat."):
        label = 0
    elif p.name.startswith("dog."):
        label = 1
    else:
        continue

    rows.append(
        {
            "image": p.name,
            "label": label,
        }
    )

processed_frame = pd.DataFrame(rows)

if processed_frame.empty:
    raise RuntimeError("No cat/dog training images found.")

processed_train_csv = DATASET_DIR / "jiaozi_train.csv"
processed_frame.to_csv(processed_train_csv, index=False)

print("\nProcessed train csv:", processed_train_csv)
print(processed_frame.head())
print("\nClass distribution:")
print(processed_frame["label"].value_counts().sort_index().to_string())

# -----------------------------
# 4. Local data info
# -----------------------------
LOCAL_DATA_INFO = {
    "benchmark": "dogs_vs_cats_redux",
    "competition": "dogs-vs-cats-redux-kernels-edition",
    "train_csv": str(processed_train_csv),
    "image_dir": str(train_dir),
    "image_column": "image",
    "label_column": "label",
    "target_column": "label",
    "image_path_template": "{image}",
    "image_extension": "",
    "num_classes": 2,
    "metric": "log_loss",
}

print("\nPrepared local Kaggle Dogs vs Cats data:")
print(json.dumps(LOCAL_DATA_INFO, indent=2, ensure_ascii=False))

# Clean old output
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

# -----------------------------
# 5. Import Jiaozi modules
# -----------------------------
from analyzer.image_statistics import ImageStatisticsAnalyzer
from features_extraction_api import module1_pipeline
from pipeline import derive_recommended_epochs, merge_modules, run_module4_generation
from recommender import OutcomeMemory, recommend
from retrieval.rag_retrieval import (
    build_all_task_lists,
    build_graph,
    build_vector_index,
    print_results,
    retrieve_top3_hybrid,
)

# -----------------------------
# 6. Module 1
# -----------------------------
print("\n[Notebook] Module 1: Parsing user requirements...")
m1_output = module1_pipeline(QUERY)

if m1_output is None:
    raise RuntimeError("Module 1 failed, so no Module 3 or Module 4 output was produced.")

# -----------------------------
# 7. Local image dataset wrapper
# -----------------------------
class LocalImageSplit:
    column_names = ["image", "label"]

    def __init__(self, frame, info):
        from PIL import Image

        self._Image = Image
        self.labels = frame[info["label_column"]].tolist()
        self.image_paths = []

        image_root = Path(info["image_dir"])
        image_column = info["image_column"]
        label_column = info["label_column"]
        template = info.get("image_path_template", "{image}")
        extension = info.get("image_extension", "")

        for _, row in frame.iterrows():
            image_value = str(row[image_column])
            relative = template.format(
                image=image_value,
                label=str(row[label_column]),
                stem=Path(image_value).stem,
            )

            image_path = image_root / relative

            if extension and not image_path.suffix:
                image_path = image_path.with_suffix(extension)

            if not image_path.is_file():
                raise FileNotFoundError(
                    f"Training image referenced by CSV is missing: {image_path}"
                )

            self.image_paths.append(image_path)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, key):
        if key == "label":
            return self.labels

        if key == "image":
            return [self._open_image(path) for path in self.image_paths]

        index = int(key)
        return {
            "image": self._open_image(self.image_paths[index]),
            "label": self.labels[index],
        }

    def _open_image(self, path):
        with self._Image.open(path) as image:
            return image.convert("RGB")


def build_local_module2_dataset(info):
    frame = pd.read_csv(info["train_csv"])

    required = {info["image_column"], info["label_column"]}
    missing = required - set(frame.columns)

    if missing:
        raise ValueError(f"Missing required CSV columns: {sorted(missing)}")

    return {"train": LocalImageSplit(frame, info)}


# -----------------------------
# 8. Module 2
# -----------------------------
print("\n[Notebook] Module 2: Analyzing sampled real Dogs vs Cats images...")
m2_report = ImageStatisticsAnalyzer().analyze(
    build_local_module2_dataset(LOCAL_DATA_INFO)
)

# -----------------------------
# 9. Merge M1 + M2
# -----------------------------
m3_input = merge_modules(m1_output, m2_report)

# Force benchmark-specific settings
m3_input["evaluation_metric"] = "log_loss"
m3_input["num_classes"] = 2

print(
    f"\n[Notebook] Merged: task={m3_input['task_type']} "
    f"size={m3_input['data_size']} "
    f"classes={m3_input.get('num_classes')} "
    f"metric={m3_input['evaluation_metric']}"
)

# -----------------------------
# 10. Module 3
# -----------------------------
print("\n[Notebook] Module 3: Retrieving model configurations...")
graph = build_graph()
collection = build_vector_index()
recommendations = retrieve_top3_hybrid(m3_input, graph, collection)

print_results(m3_input, recommendations, graph)

try:
    recommendations = recommend(
        recommendations,
        m2_report,
        m3_input,
        memory=OutcomeMemory(),
    )
    print("[Notebook] Recommender re-ranked candidates.")
except Exception as exc:
    print(f"[Notebook] Recommender skipped: {exc}")

# -----------------------------
# 11. Build task lists and inject Dogs vs Cats settings
# -----------------------------
task_lists = build_all_task_lists(recommendations, graph, fmt="nl")

for task_list in task_lists:
    model_config = task_list.get("model_config")

    if not isinstance(model_config, dict):
        continue

    model_config.update(
        {
            "num_classes": LOCAL_DATA_INFO["num_classes"],
            "train_csv": LOCAL_DATA_INFO["train_csv"],
            "image_dir": LOCAL_DATA_INFO["image_dir"],
            "image_column": LOCAL_DATA_INFO["image_column"],
            "label_column": LOCAL_DATA_INFO["label_column"],
            "target_column": LOCAL_DATA_INFO["target_column"],
            "image_path_template": LOCAL_DATA_INFO["image_path_template"],
            "image_extension": LOCAL_DATA_INFO["image_extension"],
            "evaluation_metric": LOCAL_DATA_INFO["metric"],
            "metric": LOCAL_DATA_INFO["metric"],
            "metric_name": LOCAL_DATA_INFO["metric"],
            "offline_smoke": not REAL_TRAINING,
            "benchmark_key": LOCAL_DATA_INFO["benchmark"],
            "competition": LOCAL_DATA_INFO["competition"],
        }
    )

    model_config.setdefault(
        "recommended_epochs",
        derive_recommended_epochs(
            m3_input.get("data_size", "medium"),
            model_config.get("finetune_strategy"),
            bool(model_config.get("pretrained_hf_id")),
        ),
    )

print("\nTop model config preview:")
print(json.dumps(task_lists[0]["model_config"], indent=2)[:2500])

# -----------------------------
# 12. Module 4 code generation
# -----------------------------
print("\n[Notebook] Module 4: Generating real-training project...")

module4 = run_module4_generation(
    task_lists,
    OUTPUT_DIR,
    skip_smoke=REAL_TRAINING,
    timeout=120,
    llm_provider="openai",
)

summary_path = OUTPUT_DIR / "module4_summary.json"

if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    raise FileNotFoundError(
        f"Module 4 summary was not generated at {summary_path}. Available files: {available}"
    )

DATASET = LOCAL_DATA_INFO["train_csv"]

print("\nModule 4 summary:", summary_path)
print("Generated project output dir:", OUTPUT_DIR)
print("DATASET:", DATASET)


Current working directory: /content/Jiaozi
Using pipeline: /content/Jiaozi/pipeline.py
DATASET_DIR: /content/data/dogs-vs-cats-redux-kernels-edition
Top-level files:
 - sample_submission.csv
 - test
 - test.zip
 - train
 - train.zip
Train image dir: /content/data/dogs-vs-cats-redux-kernels-edition/train/train
Sample train files: ['cat.0.jpg', 'cat.1.jpg', 'cat.10.jpg', 'cat.100.jpg', 'cat.1000.jpg', 'cat.10000.jpg', 'cat.10001.jpg', 'cat.10002.jpg', 'cat.10003.jpg', 'cat.10004.jpg']

Processed train csv: /content/data/dogs-vs-cats-redux-kernels-edition/jiaozi_train.csv
          image  label
0     cat.0.jpg      0
1     cat.1.jpg      0
2    cat.10.jpg      0
3   cat.100.jpg      0
4  cat.1000.jpg      0

Class distribution:
label
0    12500
1    12500

Prepared local Kaggle Dogs vs Cats data:
{
  "benchmark": "dogs_vs_cats_redux",
  "competition": "dogs-vs-cats-redux-kernels-edition",
  "train_csv": "/content/data/dogs-vs-cats-redux-kernels-edition/jiaozi_train.csv",
  "image_dir": "/

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Index] Added 18 backbone entries and refreshed 18 total entries.

──────────────────────────────────────────────────────────────────────
Task   : classification
Input  : size=large  priority=balanced
Desc   : Binary image classification for cats and dogs. Images are natural photos of cats and dogs from the Kaggle Dogs vs Cats Redux competition. The target label is binary: 0 for cat and 1 for dog. The primary metric is binary log loss, and the model should output calibrated probabilities for the dog class. Also report accuracy and ROC-AUC if available. Dataset properties: natural animal images, varied backgrounds, poses, lighting, and image sizes. Resource constraint: single Colab T4 GPU. Model constraint: use only publicly accessible pretrained checkpoints that do not require gated access. The agent should autonomously select the model, preprocessing, augmentation, loss, optimizer, and validation strategy. The best checkpoint should be selected by validation log loss or validation ROC

In [8]:
summary_path = OUTPUT_DIR / 'module4_summary.json'
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    print(f'Module 4 summary is missing: {summary_path}')
    print('Available files:', available)
else:
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2, ensure_ascii=False))

print('\nGenerated files:')
if OUTPUT_DIR.exists():
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(path.name)
else:
    print('Output directory was not created.')


{
  "candidates": [
    {
      "backbone": "dinov2",
      "finetune_strategy": "full",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 1,
      "score": 1.0,
      "task_type": "classification"
    },
    {
      "backbone": "dinov3",
      "finetune_strategy": "partial",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 2,
      "score": 0.627,
      "task_type": "classification"
    },
    {
      "backbone": "dinov3",
      "finetune_strategy": "partial",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 3,
      "score": 0.627,
      "task_type": "classification"
    },
    {
      "backbone": "resnet",
      "finetune_strategy": "full",
      "loss": "cross_entropy_loss",
      "optimizer": "adam",
      "rank": 4,
      "score": 0.45,
      "task_type": "classification"
    }
  ],
  "errors": [],
  "generated_files": [
    "README_generated.md",
    "configs.json",
    "evaluate.py",
    "gene

In [9]:
# !grep -n "epoch.*loss\|val_.*\|print" /content/jiaozi_generated_pipeline/train.py | head -100

# Patch train.py: add flush=True without breaking __future__ imports

from pathlib import Path
import re

train_path = Path("/content/jiaozi_generated_pipeline/train.py")
text = train_path.read_text(encoding="utf-8")

if "JIAOZI_SAFE_FLUSH_PATCH" in text:
    print("train.py already safely patched.")
else:
    lines = text.splitlines()

    insert_at = 0

    # Keep the shebang/encoding/module docstring/__future__ imports at the very beginning.
    # Find the last `from __future__ import ...` and then insert the patch.
    for i, line in enumerate(lines):
        if line.startswith("from __future__ import"):
            insert_at = i + 1

    safe_patch = [
        "",
        "# JIAOZI_SAFE_FLUSH_PATCH",
        "import functools as _jiaozi_functools",
        "import builtins as _jiaozi_builtins",
        "print = _jiaozi_functools.partial(_jiaozi_builtins.print, flush=True)",
        "",
    ]

    lines = lines[:insert_at] + safe_patch + lines[insert_at:]
    train_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

    print("Safely patched train.py. print() will flush immediately.")


Safely patched train.py. print() will flush immediately.


## Train the recommended model (real data)

This step only runs when `REAL_TRAINING = True`. It executes the generated `run.py`,
which loads the real dataset, trains the model Module 3 recommended, evaluates on the
test split, and saves checkpoints. Select a GPU runtime first for reasonable speed.


In [10]:
import json
import os
import subprocess
import sys
from pathlib import Path

if not REAL_TRAINING:
    print('REAL_TRAINING is False - skipping the real training step.')
    print('Set REAL_TRAINING = True in the run cell above to train on real data.')
else:
    # Persist checkpoints on Google Drive so they survive runtime recycling, and
    # resume automatically if a previous run left one. Set False to keep them in
    # the ephemeral generated project dir instead.
    SAVE_CHECKPOINTS_TO_DRIVE = True

    cfg_path = OUTPUT_DIR / 'configs.json'
    configs = json.loads(cfg_path.read_text(encoding='utf-8'))
    cfg = configs[0]
    mc = cfg.get('model_config', cfg)

    epochs = EPOCHS
    if epochs is None:
        epochs = mc.get('recommended_epochs') or cfg.get('recommended_epochs') or 10
    backbone = mc.get('backbone') or cfg.get('backbone')
    dataset_used = mc.get('dataset_id') or cfg.get('dataset_id') or DATASET

    if SAVE_CHECKPOINTS_TO_DRIVE and Path("/content/drive/MyDrive").exists():
        tag = f"dogs_vs_cats_{backbone}_fresh"
        ckpt_dir = f"/content/drive/MyDrive/Jiaozi/checkpoints/{tag}"

        import shutil

        if Path(ckpt_dir).exists():
            print("Removing old checkpoint dir:", ckpt_dir)
            shutil.rmtree(ckpt_dir)

        os.makedirs(ckpt_dir, exist_ok=True)

        cfg["checkpoint_dir"] = ckpt_dir
        cfg["resume_checkpoint"] = None

        cfg_path.write_text(
            json.dumps(configs, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

        print("Checkpoints ->", ckpt_dir, "(resume: disabled, fresh training)")

    elif SAVE_CHECKPOINTS_TO_DRIVE:
        print('WARNING: Drive not mounted (run the Drive cell above); checkpoints will be ephemeral.')

    print(f'Training {backbone} for {epochs} epochs on {dataset_used} ...')
    train_command = [sys.executable, '-u', 'run.py', '--epochs', str(epochs)]
    print('Command:', ' '.join(train_command), '(cwd:', OUTPUT_DIR, ')\n')
    result = subprocess.run(
    train_command,
    cwd=str(OUTPUT_DIR),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(f"Training failed with return code {result.returncode}")
    print('\nReal training finished. Checkpoints under:', cfg.get('checkpoint_dir', 'checkpoints (ephemeral)'))


Removing old checkpoint dir: /content/drive/MyDrive/Jiaozi/checkpoints/dogs_vs_cats_dinov2_fresh
Checkpoints -> /content/drive/MyDrive/Jiaozi/checkpoints/dogs_vs_cats_dinov2_fresh (resume: disabled, fresh training)
Training dinov2 for 15 epochs on /content/data/dogs-vs-cats-redux-kernels-edition/jiaozi_train.csv ...
Command: /usr/bin/python3 -u run.py --epochs 15 (cwd: /content/jiaozi_generated_pipeline )


Loading weights: 100%|██████████| 223/223 [00:09<00:00, 24.74it/s]
[train] requested_backbone=dinov2 hf_id=facebook/dinov2-base source=huggingface actual_model=facebook/dinov2-base backbone_class=_HFBackbone feature_pooling=pooler_output_or_cls_token
[train] params total=86582018 trainable=86582018 backbone_trainable=86580480 head_trainable=1538 other_trainable=0
[train] finetune strategy=full unfreeze_last_n_blocks=0 frozen_backbone_param_tensors=0 partial_unfrozen_param_tensors=0
[train] epoch 1/15  loss=0.0457  val_log_loss=0.0289  val_acc=0.9934  lr=9.90e-06  steps=625  time=348

## Show the result

Reads the best checkpoint and prints its validation score **instantly** (the metric was
already computed during training — no re-evaluation). Set `FULL_EVAL = True` for a full
re-evaluation (macro-F1, params, sample count), which costs one extra pass over the eval set.


In [11]:
import json, torch
from pathlib import Path

# Locate best_model.pt (prefer the Drive checkpoint dir set by the training cell).
_candidates = []
try:
    _candidates.append(Path(ckpt_dir) / 'best_model.pt')
except NameError:
    pass
_candidates.append(Path(OUTPUT_DIR) / 'checkpoints' / 'best_model.pt')
best_path = next((p for p in _candidates if p.exists()), None)
if best_path is None:
    raise FileNotFoundError(f'best_model.pt not found in: {[str(p) for p in _candidates]}')

ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
print('=== RESULT (best checkpoint) ===')
print('checkpoint :', best_path)
print('best_epoch :', ckpt.get('best_epoch'))
print('best_metric:', ckpt.get('best_metric'), '(validation metric at best epoch — no re-eval)')
print('validation :', json.dumps(ckpt.get('validation'), ensure_ascii=False))

# Full re-evaluation (macro_f1, params, num_samples) — one extra pass over the eval set.
FULL_EVAL = False
if FULL_EVAL:
    import os, sys
    os.chdir(OUTPUT_DIR); sys.path.insert(0, str(OUTPUT_DIR))
    from model import build_model
    from evaluate import evaluate
    _cfg = json.loads((Path(OUTPUT_DIR) / 'configs.json').read_text(encoding='utf-8'))[0]
    _cfg.update(_cfg.pop('model_config', {}) or {})
    _model = build_model(_cfg); _model.load_state_dict(ckpt['model_state_dict'])
    print('\n=== FULL EVALUATE ===')
    print(json.dumps(evaluate(_model, _cfg), indent=2, ensure_ascii=False))


=== RESULT (best checkpoint) ===
checkpoint : /content/drive/MyDrive/Jiaozi/checkpoints/dogs_vs_cats_dinov2_fresh/best_model.pt
best_epoch : 14
best_metric: 0.019187405256742018 (validation metric at best epoch — no re-eval)
validation : {"metric_name": "log_loss", "metric_value": 0.019187405256742018, "accuracy": 0.996999979019165, "epoch": 14}


In [12]:
#  Generate Kaggle Dogs vs Cats submission.csv

import os
import sys
import json
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image

OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")
DATA_DIR = Path("/content/data/dogs-vs-cats-redux-kernels-edition")

sample_path = DATA_DIR / "sample_submission.csv"
submission_path = OUTPUT_DIR / "submission.csv"

assert OUTPUT_DIR.exists(), OUTPUT_DIR
assert DATA_DIR.exists(), DATA_DIR
assert sample_path.exists(), sample_path

# Locate test images
candidate_test_dirs = [
    DATA_DIR / "test",
    DATA_DIR / "test" / "test",
]

test_dir = None
for d in candidate_test_dirs:
    if d.exists() and list(d.glob("*.jpg")):
        test_dir = d
        break

if test_dir is None:
    matches = []
    for d in DATA_DIR.rglob("*"):
        if d.is_dir():
            jpgs = list(d.glob("*.jpg"))
            if jpgs and any(p.stem.isdigit() for p in jpgs[:50]):
                matches.append(d)
    if matches:
        test_dir = matches[0]

if test_dir is None:
    raise FileNotFoundError(f"Could not find test images under {DATA_DIR}")

print("Test image dir:", test_dir)

os.chdir(OUTPUT_DIR)
if str(OUTPUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUTPUT_DIR))

from model import build_model
from train import _build_image_transform

# -----------------------------
# 1. Load config
# -----------------------------
configs = json.loads((OUTPUT_DIR / "configs.json").read_text(encoding="utf-8"))
cfg = configs[0]

if "model_config" in cfg:
    flat_cfg = {**cfg, **cfg["model_config"]}
else:
    flat_cfg = cfg

flat_cfg["num_classes"] = 2
flat_cfg["metric_name"] = "log_loss"
flat_cfg["evaluation_metric"] = "log_loss"

print("Config preview:")
for k in ["backbone", "pretrained_hf_id", "model_name", "num_classes", "metric_name"]:
    if k in flat_cfg:
        print(k, ":", flat_cfg[k])

# -----------------------------
# 2. Find best checkpoint
# -----------------------------
ckpt_candidates = []

try:
    ckpt_candidates.append(Path(ckpt_dir) / "best_model.pt")
except NameError:
    pass

if flat_cfg.get("checkpoint_dir"):
    ckpt_candidates.append(Path(flat_cfg["checkpoint_dir"]) / "best_model.pt")

ckpt_candidates.append(OUTPUT_DIR / "checkpoints" / "best_model.pt")
ckpt_candidates.append(OUTPUT_DIR / "best_model.pt")

best_path = next((p for p in ckpt_candidates if p.exists()), None)

if best_path is None:
    raise FileNotFoundError(
        "best_model.pt not found. Checked:\n" + "\n".join(str(p) for p in ckpt_candidates)
    )

print("Using checkpoint:", best_path)

# -----------------------------
# 3. Load model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_model(flat_cfg).to(device)

ckpt = torch.load(best_path, map_location=device, weights_only=False)

if "model_state_dict" in ckpt:
    state_dict = ckpt["model_state_dict"]
elif "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

model.load_state_dict(state_dict)
model.eval()

# -----------------------------
# 4. Transform
# -----------------------------
transform = _build_image_transform(flat_cfg, "test")

# -----------------------------
# 5. Read sample submission ids
# -----------------------------
sample = pd.read_csv(sample_path)

print("sample_submission columns:", list(sample.columns))
assert "id" in sample.columns, sample.columns

test_ids = sample["id"].astype(str).tolist()

# -----------------------------
# 6. Predict dog probability
# -----------------------------
rows = []
batch = []
batch_ids = []

def logits_to_dog_prob(logits):
    """
    Output must be probability that the image is dog.
    Training label convention:
    cat -> 0
    dog -> 1
    """
    if logits.ndim == 1:
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        return probs

    if logits.shape[1] == 1:
        probs = torch.sigmoid(logits[:, 0]).detach().cpu().numpy()
        return probs

    probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
    return probs[:, 1]

def flush():
    global batch, batch_ids, rows

    if not batch:
        return

    x = torch.stack(batch).to(device)

    with torch.no_grad():
        out = model(x)

        if isinstance(out, dict):
            logits = out.get("logits", next(iter(out.values())))
        elif isinstance(out, (tuple, list)):
            logits = out[0]
        else:
            logits = out

        dog_probs = logits_to_dog_prob(logits)

    for image_id, prob in zip(batch_ids, dog_probs):
        # Clip to avoid extreme 0/1 probabilities for log loss submission
        prob = float(np.clip(prob, 1e-5, 1 - 1e-5))
        rows.append(
            {
                "id": int(image_id),
                "label": prob,
            }
        )

    batch = []
    batch_ids = []

for image_id in test_ids:
    img_path = test_dir / f"{image_id}.jpg"

    if not img_path.exists():
        raise FileNotFoundError(img_path)

    img = Image.open(img_path).convert("RGB")
    batch.append(transform(img))
    batch_ids.append(image_id)

    if len(batch) >= 64:
        flush()

flush()

# -----------------------------
# 7. Save submission
# -----------------------------
sub = pd.DataFrame(rows)

# Keep same order as sample_submission
sample_order = sample[["id"]].copy()
sample_order["id"] = sample_order["id"].astype(int)

sub = sample_order.merge(sub, on="id", how="left")

assert len(sub) == len(sample), f"submission rows {len(sub)} != sample rows {len(sample)}"
assert sub["label"].notna().all(), "Some predictions are missing."
assert sub["label"].between(0, 1).all(), "Probabilities must be in [0, 1]."

sub.to_csv(submission_path, index=False)

print("Wrote submission:", submission_path)
print("Shape:", sub.shape)
print("Prediction summary:")
print(sub["label"].describe())
display(sub.head())


Test image dir: /content/data/dogs-vs-cats-redux-kernels-edition/test/test
Config preview:
backbone : dinov2
pretrained_hf_id : facebook/dinov2-base
num_classes : 2
metric_name : log_loss
Using checkpoint: /content/drive/MyDrive/Jiaozi/checkpoints/dogs_vs_cats_dinov2_fresh/best_model.pt


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

sample_submission columns: ['id', 'label']
Wrote submission: /content/jiaozi_generated_pipeline/submission.csv
Shape: (12500, 2)
Prediction summary:
count    12500.000000
mean         0.501431
std          0.499766
min          0.000010
25%          0.000010
50%          0.992054
75%          0.999990
max          0.999990
Name: label, dtype: float64


,id,label
0,1,0.99999
1,2,0.99999
2,3,0.99999
3,4,0.99999
4,5,0.00001


In [13]:
!kaggle competitions submit \
  -c dogs-vs-cats-redux-kernels-edition \
  -f /content/jiaozi_generated_pipeline/submission.csv \
  -m "Jiaozi Dogs vs Cats Redux late submission3"


100% 313k/313k [00:00<00:00, 1.71MB/s]
Successfully submitted to Dogs vs. Cats Redux: Kernels Edition

In [14]:
!kaggle competitions submissions -c dogs-vs-cats-redux-kernels-edition


     ref  fileName        date                        description                                 status                     publicScore  privateScore  
--------  --------------  --------------------------  ------------------------------------------  -------------------------  -----------  ------------  
54621030  submission.csv  2026-07-12 20:18:29.650000  Jiaozi Dogs vs Cats Redux late submission2  SubmissionStatus.COMPLETE  0.06594      0.06594       
54607558  submission.csv  2026-07-12 12:24:12.100000  Jiaozi Dogs vs Cats Redux late submission2  SubmissionStatus.COMPLETE  0.07270      0.07270       
